In [ ]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)


In [ ]:
import plot
from read_chroma import read_only_chroma, read_chromato_and_chromato_cube
import matplotlib.pyplot as plt
from identification import compute_matches_identification, cohort_identification_alignment_input_format_txt, cohort_identification_to_csv, sample_identification
from matching import matching_nist_lib_from_chromato_cube
import projection
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

import mass_spec
from sklearn.cluster import DBSCAN
from scipy.signal import savgol_filter
from skimage.restoration import estimate_sigma
from dbscan_peak import detection_mass_par_mass_Dog
import dbscan_peak

In [ ]:
filename="D:/GCxGC_MS/DATA/Dossier_partagé_GCxGC/Manue/GCxGC_VOLATIL_CF_08bis_postPTR/15-04-25_817822_QC_23newEI.cdf"
filename = "/home/camille/Documents/app/data/J-A-034-751325-Tedlar.h5"
chromato_tic, time_rn, chromato_cube, sigma, mass_range=read_chromato_and_chromato_cube(filename, mod_time=1.7,pre_process=False)

In [ ]:
coordinates, spec_list, area = dbscan_peak.detection_mass_par_mass_Dog(chromato_cube,(chromato_tic, time_rn),
                                                            1.7,
                                                                abs_threshold=500,
                                                                rel_threshold=0.5,
                                                                noise_factor=3,
                                                                min_sigma=1,
                                                                max_sigma=20,
                                                                sigma_ratio=2,
                                                                overlap=0.5, 
                                                                max_peak_per_mass=600,
                                                                rt1_delta=1, 
                                                                rt2_delta=0.01,
                                                                min_size_cluster_mass=3, 
                                                                thr_debscan=0.01, 
                                                                multi_processing=True,
                                                                cleaning_close_peak=True)




In [ ]:
coordinates[0]

In [ ]:
plt.rcParams['figure.figsize'] = [30, 5]
coordinates_in_chromato=projection.matrix_to_chromato(coordinates,  time_rn, 1.7, chromato_tic.shape)
plot.visualizer2((chromato_tic, time_rn), title="chromato", log_chromato=True,  mod_time=1.7,points=coordinates_in_chromato)

In [ ]:
i=1
plot.visualizer2((chromato_tic, time_rn), title="chromato", log_chromato=True,  mod_time=1.7,points=coordinates_in_chromato,
                 rt1=coordinates_in_chromato[i,0],rt2=coordinates_in_chromato[i,1],rt1_window=0.2,rt2_window=1.5)

In [ ]:
plt.plot(chromato_tic[coordinates[1][0],:])

In [ ]:
import matching
matches = matching.matching_nist_lib_from_chromato_cube(
            (chromato_tic, time_rn, mass_range), chromato_cube, coordinates,
            mod_time=1.7,
            match_factor_min=700, nist=False)
coordinates_in_chromato=projection.matrix_to_chromato(coordinates, time_rn, 1.7, chromato_tic.shape)

In [ ]:
import h5py
import netCDF4 as nc
def get_scan_number(file_path):
    """Get scan number from file."""
    try:
        if file_path.endswith((".h5", ".H5")):
            with h5py.File(file_path, 'r') as f:
                return f.attrs['scan_number_size']
        elif file_path.endswith((".cdf", ".CDF")):
            with nc.Dataset(file_path, 'r') as dt:
                return dt.dimensions['scan_number'].size
        else:
            raise ValueError("Unsupported file format. Please provide a .h5 or .cdf file.")
    except Exception as e:
        raise ValueError(f"Error while reading file {file_path}: {e}")

def get_mod_time(file_path):
    """Get modulation time based on scan_number from file."""
    scan_number = get_scan_number(file_path)
    modulation_times = {
        328125: (1.25, "G0/plasma"),
        540035: (1.7, "exhaled air")
    }
    if scan_number in modulation_times:
        mod_time, data_type = modulation_times[scan_number]
        print(f"   Data type: {data_type}")
        return mod_time
    else:
        print(f"   ⚠️  Unknown scan_number: {scan_number}, using default modulation time")
        return

In [ ]:
base_name = os.path.splitext(os.path.basename(filename))[0]
formated_spectra=True
quant ="masse"
extract_patch=False
output_hdf5_file=None

In [ ]:

matches_identification, sample_metadata_list = compute_matches_identification(
                matches, spec_list, area, chromato_tic, chromato_cube,time_rn,get_mod_time(filename), mass_range,base_name,
                formated_spectra, quant, extract_patch, output_hdf5_file, integration_mod_max1=True, integration_mod_max2=False)

In [ ]:
matches_identification, sample_metadata_list = compute_matches_identification(
                matches, spec_list, area, chromato_tic, chromato_cube,time_rn,get_mod_time(filename), mass_range,base_name,
                formated_spectra, quant, extract_patch, output_hdf5_file, integration_mod_max1=False, integration_mod_max2=True)